# GraviLens v2 driver

Thin Colab driver only. All simulation, training, evaluation, and audit logic
lives in the `gravilens` package.

Set `REPO` and `TAG` below. The tag pins the exact audited code: push it with
`git push origin main --tags`. Bump `TAG` (and the tag itself) if the package
changes. A GPU runtime is strongly recommended for the ablation; data generation
is CPU-bound.

In [ ]:
import os
REPO = "https://github.com/<handle>/gravilens"
TAG = "v0.1.0"
if not os.path.isdir('/content/gravilens'):
    !git clone {REPO} /content/gravilens
%cd /content/gravilens
!git fetch --tags && git checkout {TAG}
!pip -q install -e ".[sim,dev]"

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## Data

Two ways to get the dataset. Run **one** of the two cells below; the cell after
them picks the right root automatically.

**Path A (fast, default): use the uploaded shards.** Upload `gravilens-v2.tar`
(the contents of `data/gravilens-v2/`, ~300 MB) to `MyDrive` first. The dataset
fingerprint is content-based, so any copy of the same data is accepted.

**Path B (from scratch, reproducible): regenerate.** ~5 CPU-hours on this machine.
Set `REGENERATE=True` in the cell below. The pixels are bit-identical to Path A's
dataset, because rendering is deterministic per triplet seed.

In [ ]:
# Path A: use the uploaded dataset. Needs gravilens-v2.tar in MyDrive.
import subprocess
from google.colab import drive

drive.mount('/content/drive')
TAR = '/content/drive/MyDrive/gravilens-v2.tar'
if not os.path.isdir('/content/gravilens-v2'):
    subprocess.run(['tar', '-xf', TAR, '-C', '/content/'], check=True)
print('uploaded dataset:', os.path.isdir('/content/gravilens-v2'))

In [ ]:
%%bash
# Path B: regenerate from scratch. Set to False to skip.
REGENERATE=False
if [ "$REGENERATE" = "True" ]; then
  python -m gravilens generate --config configs/data.yaml --splits train > train_gen.log 2>&1 &
  python -m gravilens generate --config configs/data.yaml --splits val   > val_gen.log   2>&1 &
  python -m gravilens generate --config configs/data.yaml --splits test  > test_gen.log  2>&1 &
  wait
  tail -n 2 train_gen.log val_gen.log test_gen.log
fi

In [ ]:
# One root for every later command: the uploaded copy if it is there,
# otherwise the regenerated one.
UPLOADED = '/content/gravilens-v2'
REGENERATED = '/content/gravilens/data/gravilens-v2'
DATA_ROOT = UPLOADED if os.path.isdir(UPLOADED) else REGENERATED
print('DATA_ROOT =', DATA_ROOT)

!python -m gravilens.dataset --data-root {DATA_ROOT} --split all

## Smoke test (2 epochs)

This checks the full train/evaluate path before spending GPU time on the 21-run
grid. The smoke artifacts go to `runs-smoke/`, not `runs/`, so they can never be
mistaken for grid results.

In [ ]:
!python -m gravilens train --config configs/ablation.yaml --data-root {DATA_ROOT} \
    --model gravilens --variant full --seed 42 --epochs 2 --out runs-smoke/gravilens_full_seed42
!python -m gravilens evaluate --data-root {DATA_ROOT} --run runs-smoke/gravilens_full_seed42

## Full ablation (21 runs)

4 variants × 3 seeds + 3 parameter-matched baselines × 3 seeds, run in the
pre-registered order: the reviewer-critical variants first, the baselines second,
`dist` last. `ablate` skips a run only when it used the same recipe and the same
dataset, and redoes it otherwise. `audit` exits non-zero until all 21 runs exist
and every run used the pre-registered recipe; `figures` writes the main table and
figures from the artifacts only.

Runtimes are ephemeral: rerun the cell to continue (finished runs are skipped).
Copy `runs/` to Drive before disconnecting.

In [ ]:
!python -m gravilens ablate --config configs/ablation.yaml --data-root {DATA_ROOT} --variants full no_sis no_sup
!python -m gravilens ablate --config configs/ablation.yaml --data-root {DATA_ROOT} --baselines vit cait vitsd
!python -m gravilens ablate --config configs/ablation.yaml --data-root {DATA_ROOT} --variants dist
!python -m gravilens audit
!python -m gravilens figures

## Save artifacts

Colab runtimes are ephemeral. Copy `runs/` to Drive or download the run directories
before disconnecting.

In [ ]:
!mkdir -p /content/drive/MyDrive/gravilens-runs
!cp -r runs /content/drive/MyDrive/gravilens-runs